# CrimeScope ML — 05 (UK). Score & Serve

Loads `@champion` models from UC, scores every LSOA + MSOA for the latest
mature month, computes blended 0–100 risk scores + tiers, and writes Delta
tables consumed by notebook `06`.

In [ ]:
spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")

In [ ]:
%pip install -q shap lightgbm
dbutils.library.restartPython()

In [ ]:
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.lightgbm
import shap
from datetime import datetime, timezone
from scipy.stats import percentileofscore

mlflow.set_registry_uri("databricks-uc")

FEATURE_LABELS = {
    "lag_1m_count": "Last month's crime count",
    "rolling_mean_3m": "3-month crime average",
    "rolling_mean_12m": "12-month crime average",
    "rolling_std_6m": "6-month crime volatility",
    "violent_ratio": "Violent crime proportion",
    "violent_ratio_6m": "6-month violent ratio",
    "imd_score": "Index of Multiple Deprivation",
    "imd_decile": "IMD decile (1=most deprived)",
    "log_pop": "Population (log scale)",
    "crime_rate_per_1k": "Crime rate per 1,000 residents",
    "month_of_year": "Month of year (seasonality)",
    "same_month_last_year": "Same month last year",
    "yoy_change": "Year-over-year change",
    "trend_3m": "3-month trend direction",
}


def blended_score(p):
    pct = np.array([percentileofscore(p, v, kind="rank") for v in p])
    log_p = np.log1p(p)
    lo, hi = log_p.min(), log_p.max()
    abs_s = (log_p - lo) / (hi - lo) * 100 if hi > lo else np.full_like(log_p, 50.0)
    return np.round(0.7 * pct + 0.3 * abs_s, 1)


def to_tier(s):
    return pd.cut(s, bins=[-0.1, 25, 50, 75, 90, 100],
                  labels=["Low", "Moderate", "Elevated", "High", "Critical"]).astype(str)


def top_drivers_json(shap_row, feat_names, feat_vals, n=5):
    idxs = np.abs(shap_row).argsort()[-n:][::-1]
    return json.dumps([
        {
            "feature": feat_names[i],
            "label": FEATURE_LABELS.get(feat_names[i], feat_names[i].replace("_", " ").title()),
            "shap_value": round(float(shap_row[i]), 4),
            "feature_value": round(float(feat_vals[i]), 4),
            "direction": "up" if shap_row[i] > 0 else "down",
        } for i in idxs
    ])

---
## Score LSOA

In [ ]:
model_lsoa = mlflow.lightgbm.load_model("models:/varanasi.default.crimescope_uk_risk_model_lsoa@champion")
try:
    model_lsoa_v = mlflow.lightgbm.load_model("models:/varanasi.default.crimescope_uk_risk_model_violent@champion")
except Exception:
    model_lsoa_v = None
try:
    model_lsoa_p = mlflow.lightgbm.load_model("models:/varanasi.default.crimescope_uk_risk_model_property@champion")
except Exception:
    model_lsoa_p = None

feat = spark.table("varanasi.default.uk_lsoa_features").toPandas()
feat = feat.dropna(subset=["lag_1m_count"])
feat["month_dt"] = pd.to_datetime(feat["month_start"])
latest_month = feat["month_dt"].max()
latest = feat[feat["month_dt"] == latest_month].copy()

feature_cols = list(model_lsoa.booster_.feature_name())
for c in feature_cols:
    if c not in latest.columns:
        latest[c] = 0.0
X_latest = latest[feature_cols].astype(float).fillna(0.0)

pred = np.maximum(np.expm1(np.maximum(model_lsoa.predict(X_latest), 0)), 0)
latest["predicted_next_30d"] = pred.round(2)
latest["risk_score"] = blended_score(pred)
latest["risk_tier"] = to_tier(latest["risk_score"])
latest["scored_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
latest["model_version"] = "uk_lsoa_lgbm_ensemble_v1"

if model_lsoa_v is not None:
    sub_feats_v = list(model_lsoa_v.booster_.feature_name())
    for c in sub_feats_v:
        if c not in latest.columns:
            latest[c] = 0.0
    pv = np.maximum(np.expm1(np.maximum(model_lsoa_v.predict(latest[sub_feats_v].astype(float).fillna(0.0)), 0)), 0)
    latest["predicted_violent_30d"] = pv.round(2)
    latest["violent_score"] = blended_score(pv)
else:
    latest["predicted_violent_30d"] = (latest["predicted_next_30d"] * 0.3).round(2)
    latest["violent_score"] = blended_score(latest["predicted_violent_30d"].values)

if model_lsoa_p is not None:
    sub_feats_p = list(model_lsoa_p.booster_.feature_name())
    for c in sub_feats_p:
        if c not in latest.columns:
            latest[c] = 0.0
    pp = np.maximum(np.expm1(np.maximum(model_lsoa_p.predict(latest[sub_feats_p].astype(float).fillna(0.0)), 0)), 0)
    latest["predicted_property_30d"] = pp.round(2)
    latest["property_score"] = blended_score(pp)
else:
    latest["predicted_property_30d"] = (latest["predicted_next_30d"] * 0.5).round(2)
    latest["property_score"] = blended_score(latest["predicted_property_30d"].values)

baseline = (
    0.30 * latest["rolling_mean_3m"].fillna(0)
    + 0.25 * latest["rolling_mean_12m"].fillna(0)
    + 0.20 * latest["lag_1m_count"].fillna(0)
    + 0.15 * latest["same_month_last_year"].fillna(latest["rolling_mean_12m"].fillna(0))
    + 0.10 * latest["lag_3m_count"].fillna(0)
).clip(lower=0)
latest["baseline_predicted"] = baseline.round(2)
latest["model_vs_baseline"] = (
    (latest["predicted_next_30d"] - latest["baseline_predicted"]) /
    latest["baseline_predicted"].clip(lower=0.1)
).round(3)
latest["trend_direction"] = np.where(
    latest["predicted_next_30d"] > latest["lag_1m_count"] * 1.05, "rising",
    np.where(latest["predicted_next_30d"] < latest["lag_1m_count"] * 0.95, "falling", "stable")
)

# SHAP top drivers
explainer = shap.TreeExplainer(model_lsoa)
shap_vals = explainer.shap_values(X_latest)
latest["top_drivers_json"] = [
    top_drivers_json(shap_vals[i], feature_cols, X_latest.iloc[i].values)
    for i in range(len(shap_vals))
]
# Map LSOA fields to the existing backend contract names
latest["tract_geoid"] = latest["lsoa_code"]
latest["NAMELSAD"] = latest["lsoa_code"]  # name is supplied via boundaries table

cols = [
    "tract_geoid", "NAMELSAD", "risk_score", "risk_tier",
    "predicted_next_30d", "predicted_violent_30d", "predicted_property_30d",
    "violent_score", "property_score",
    "incident_count", "y_incidents_12m", "trend_direction",
    "model_vs_baseline", "baseline_predicted", "top_drivers_json",
    "total_pop", "imd_score", "imd_decile",
    "scored_at", "model_version",
]
out = latest[[c for c in cols if c in latest.columns]]
spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_lsoa_risk_scores")
spark.sql("ALTER TABLE varanasi.default.uk_lsoa_risk_scores SET TBLPROPERTIES ('comment' = 'Latest per-LSOA risk scores + SHAP drivers (CrimeScope UK).')")
print(f"Wrote {len(out):,} LSOA scores for {latest_month.date()}")

---
## Score MSOA

In [ ]:
model_msoa = mlflow.lightgbm.load_model("models:/varanasi.default.crimescope_uk_risk_model_msoa@champion")

featM = spark.table("varanasi.default.uk_msoa_features").toPandas()
featM = featM.dropna(subset=["lag_1m_count"])
featM["month_dt"] = pd.to_datetime(featM["month_start"])
latestM = featM[featM["month_dt"] == featM["month_dt"].max()].copy()

feat_cols_m = list(model_msoa.booster_.feature_name())
for c in feat_cols_m:
    if c not in latestM.columns:
        latestM[c] = 0.0
XM = latestM[feat_cols_m].astype(float).fillna(0.0)
predM = np.maximum(np.expm1(np.maximum(model_msoa.predict(XM), 0)), 0)
latestM["predicted_next_30d"] = predM.round(2)
latestM["risk_score"] = blended_score(predM)
latestM["risk_tier"] = to_tier(latestM["risk_score"])
# Synthetic split for sub-scores until MSOA sub-models trained
latestM["predicted_violent_30d"] = (latestM["predicted_next_30d"] * 0.30).round(2)
latestM["predicted_property_30d"] = (latestM["predicted_next_30d"] * 0.50).round(2)
latestM["violent_score"] = blended_score(latestM["predicted_violent_30d"].values)
latestM["property_score"] = blended_score(latestM["predicted_property_30d"].values)
baseline = (
    0.30 * latestM["rolling_mean_3m"].fillna(0)
    + 0.25 * latestM["rolling_mean_12m"].fillna(0)
    + 0.20 * latestM["lag_1m_count"].fillna(0)
    + 0.15 * latestM["same_month_last_year"].fillna(latestM["rolling_mean_12m"].fillna(0))
    + 0.10 * latestM["lag_3m_count"].fillna(0)
).clip(lower=0)
latestM["baseline_predicted"] = baseline.round(2)
latestM["model_vs_baseline"] = (
    (latestM["predicted_next_30d"] - latestM["baseline_predicted"]) /
    latestM["baseline_predicted"].clip(lower=0.1)
).round(3)
latestM["trend_direction"] = np.where(
    latestM["predicted_next_30d"] > latestM["lag_1m_count"] * 1.05, "rising",
    np.where(latestM["predicted_next_30d"] < latestM["lag_1m_count"] * 0.95, "falling", "stable")
)

explainer_m = shap.TreeExplainer(model_msoa)
sv = explainer_m.shap_values(XM)
latestM["top_drivers_json"] = [
    top_drivers_json(sv[i], feat_cols_m, XM.iloc[i].values) for i in range(len(sv))
]
latestM["tract_geoid"] = latestM["msoa_code"]
latestM["NAMELSAD"] = latestM["msoa_code"]
latestM["scored_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
latestM["model_version"] = "uk_msoa_lgbm_ensemble_v1"

outM = latestM[[c for c in [
    "tract_geoid", "NAMELSAD", "risk_score", "risk_tier",
    "predicted_next_30d", "predicted_violent_30d", "predicted_property_30d",
    "violent_score", "property_score",
    "incident_count", "y_incidents_12m", "trend_direction",
    "model_vs_baseline", "baseline_predicted", "top_drivers_json",
    "total_pop", "imd_score", "imd_decile",
    "scored_at", "model_version",
] if c in latestM.columns]]
spark.createDataFrame(outM).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_msoa_risk_scores")
spark.sql("ALTER TABLE varanasi.default.uk_msoa_risk_scores SET TBLPROPERTIES ('comment' = 'Latest per-MSOA risk scores + SHAP drivers (CrimeScope UK).')")
print(f"Wrote {len(outM):,} MSOA scores")

---
## Append to history (audit trail)

In [ ]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS varanasi.default.uk_risk_scores_history (
    grain STRING, tract_geoid STRING, risk_score DOUBLE, risk_tier STRING,
    predicted_next_30d DOUBLE, scored_at STRING, model_version STRING
  ) USING delta
  COMMENT 'Append-only audit trail of CrimeScope UK risk scoring runs.'
""")

spark.sql("""
  INSERT INTO varanasi.default.uk_risk_scores_history
  SELECT 'lsoa', tract_geoid, risk_score, risk_tier,
         predicted_next_30d, scored_at, model_version
  FROM varanasi.default.uk_lsoa_risk_scores
""")
spark.sql("""
  INSERT INTO varanasi.default.uk_risk_scores_history
  SELECT 'msoa', tract_geoid, risk_score, risk_tier,
         predicted_next_30d, scored_at, model_version
  FROM varanasi.default.uk_msoa_risk_scores
""")
print("History updated")